In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
# Initialize tqdm with pandas
tqdm.pandas()

In [ ]:
# 2. Download Arabic Sentiment Twitter Dataset

!pip install -q kaggle


from google.colab import files
files.upload()


!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


!kaggle datasets download -d mksaad/arabic-sentiment-twitter-corpus


!unzip arabic-sentiment-twitter-corpus.zip

In [ ]:
#3. Load the Data into Pandas
import pandas as pd
# Read positive and negative training files
df_pos = pd.read_csv('train_Arabic_tweets_positive_20190413.tsv', sep='\t', header=None, names=['label', 'text'])
df_neg = pd.read_csv('train_Arabic_tweets_negative_20190413.tsv', sep='\t', header=None, names=['label', 'text'])

# Combine both datasets
df = pd.concat([df_pos, df_neg], ignore_index=True)


print(df.head())

## Text Preprocessing Pipeline
We will apply several cleaning steps suitable for Arabic text.

In [ ]:
import re
# 4. Cleaning Functions
def clean_html(raw_html):
    clean_re = re.compile('<.*?>')
    clean_text = re.sub(clean_re, '', raw_html)
    return clean_text


html_content = "<p>مرحباً بك في <b>مشروع تحليل المشاعر</b>! <br> هذا النص يحتوي على وسم.</p>"
print(clean_html(html_content))

In [ ]:
import re

def remove_urls(text):
  """Remove URLs from text"""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def final_clean(text):
    """Final text cleaning: remove extra spaces"""
    text = remove_urls(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Define sample_text before using it
sample_text = "هذا نص تجريبي يحتوي على رابط: https://example.com و مسافات    زائدة."

print(final_clean(sample_text))

هذا نص تجريبي يحتوي على رابط: و مسافات زائدة.


In [ ]:
import re

def remove_digits(text):
    # النمط [0-9] يزيل الأرقام الغربية
    # النمط [٠-٩] يزيل الأرقام العربية الشرقية (الهندية)
    pattern = r'[0-9٠١٢٣٤٥٦٧٨٩]'
    return re.sub(pattern, '', text)

# تجربة الكود
sample_text = "سعر المنتج هو 1500 ريال، والتوصيل خلال ٤ أيام."
clean_text = remove_digits(sample_text)

print(clean_text) # المخرجات: سعر المنتج هو  ريال، والتوصيل خلال  أيام.

سعر المنتج هو  ريال، والتوصيل خلال  أيام.


In [ ]:
import re

def remove_punctuation(text):
    # هذا النمط يحذف كل ما هو ليس حرفاً عربياً أو إنجليزياً أو مسافة
    # ويشمل ذلك علامات الترقيم العربية والغربية
    clean_text = re.sub(r'[^\w\s]', '', text)
    # لإزالة الشرطة السفلية (_) التي تعتبر جزءاً من \w في بعض الأحيان
    clean_text = clean_text.replace('_', '')
    return clean_text

# تجربة الكود
sample = "هل أنت مستعد للمشروع؟ (أنا متحمس جداً!)؛ لنبدأ العمل..."
print(remove_punctuation(sample))

هل أنت مستعد للمشروع أنا متحمس جدا لنبدأ العمل


In [ ]:
import re

def reduce_lengthening(text):
    # يبحث عن أي حرف مكرر أكثر من مرتين ويستبدله بحرفين فقط
    pattern = re.compile(r"(.)\1{2,}")
    return pattern.sub(r"\1\1", text)

# تجربة
print(reduce_lengthening("الجووووو جمييل جدااااا"))
# المخرجات: الجو جميل جداا

الجوو جمييل جداا


In [ ]:
def normalize_arabic(text):
    # توحيد الألفات
    text = re.sub("[إأآ]", "أ", text)
    # توحيد الياء والالف المقصورة
    text = re.sub("ى", "ي", text)
    # توحيد التاء المربوطة والهاء
    text = re.sub("ة", "ه", text)
    return text

In [ ]:
!pip install pyspellchecker
from spellchecker import SpellChecker

# ملاحظة: يجب توفير قاموس عربي أو تدريبها على كلمات المشروع
spell = SpellChecker(language=None)
# يمكنك تحميل قاموس عربي واستخدامه هنا

In [ ]:
import nltk
from nltk.corpus import stopwords

# تحميل كلمات التوقف العربية
nltk.download('stopwords')
stop_words = set(stopwords.words('arabic'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

# تجربة
sample = "هذا المشروع هو من أهم المشاريع في تحليل المشاعر"
print(remove_stopwords(sample))
# المخرجات: المشروع أهم المشاريع تحليل المشاعر

المشروع أهم المشاريع تحليل المشاعر


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# 1. خطوة التقسيم (هذه هي الخطوة التي كانت تنقصك في الخلية الحالية)
# تأكدي أن df['clean_text'] و df['label'] موجودين بالفعل
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. إعداد الـ Vectorizer (N-Gram)
cv_ngram = CountVectorizer(ngram_range=(1, 2), max_features=10000)

# 3. الملاءمة والتحويل لبيانات التدريب فقط
X_train_cv = cv_ngram.fit_transform(X_train)

# 4. تحويل بيانات الاختبار بناءً على ما تعلمه من التدريب
X_test_cv = cv_ngram.transform(X_test)

print(f"✅ تم تعريف المتغيرات بنجاح.")
print(f"شكل مصفوفة التدريب: {X_train_cv.shape}")

✅ تم تعريف المتغيرات بنجاح.
شكل مصفوفة التدريب: (36220, 10000)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# إنشاء وتدريب الموديل
rf_cv = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_cv.fit(X_train_cv, y_train)

# التنبؤ والتقييم
y_pred_cv = rf_cv.predict(X_test_cv)

print("--- نتائج Random Forest باستخدام Count Vectorizer & N-Grams ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_cv):.4f}")
print(classification_report(y_test, y_pred_cv))

--- نتائج Random Forest باستخدام Count Vectorizer & N-Grams ---
Accuracy: 0.7671
              precision    recall  f1-score   support

         neg       0.75      0.80      0.77      4503
         pos       0.79      0.73      0.76      4552

    accuracy                           0.77      9055
   macro avg       0.77      0.77      0.77      9055
weighted avg       0.77      0.77      0.77      9055

